# Orbital optimization for quantum chemistry

The quantum chemistry hamiltonian depends on the choice of orbitals. If the MPS were exact that choice wouldn't matter, but at finite bond dimension (or with frozen orbitals) it does. Rotating the orbitals by an orthogonal $U$ changes the integrals
$$K \to U^T K U,\qquad V_{pqrs} \to \sum V_{abcd}\,U_{ap}U_{bq}U_{cr}U_{ds},$$
and the energy
$$E(\psi,U) = E_0 + K(U)\cdot dK(\psi) + V(U)\cdot dV(\psi)$$
is linear in the integrals, with $dK$, $dV$ the one- and two-body reduced density matrices of $\psi$.

In this notebook I demonstrate some (claude updated) algorithms that co-optimize the orbitals with the MPS
Everything runs on N₂ in STO-3G (10 orbitals, 14 electrons, FCI energy −107.64825097).

In [1]:
using LinearAlgebra, Random, MPSKit, TensorKit, MPSKitExperimental
LinearAlgebra.BLAS.set_num_threads(1);

(ERI, K, E0, NORB, NELEC, MS2) = parse_fcidump("N2.STO3G.FCIDUMP")
ints = QChemIntegrals(E0, K, ERI)   # full-space MO integrals, the reference orbitals
const E_FCI = -107.64825097

psp = Vect[(Irrep[U₁] ⊠ Irrep[SU₂] ⊠ FermionParity)]((0, 0, 0) => 1, (1, 1 // 2, 1) => 1, (2, 0, 0) => 1);

# a random mps on the orbitals `active`, the orbitals before `active` are doubly occupied
function random_mps(active = 1:NORB; seed = 1)
    Random.seed!(seed)
    ne = NELEC - 2 * (first(active) - 1)
    left = Vect[(Irrep[U₁] ⊠ Irrep[SU₂] ⊠ FermionParity)]((-ne, MS2 // 2, mod(-ne, 2)) => 1)
    virtual = Vect[(Irrep[U₁] ⊠ Irrep[SU₂] ⊠ FermionParity)]((i, s, b) => 1 for i in -ne:0, s in 0:(1 // 2):(MS2 // 2 + 1), b in (0, 1))
    FiniteMPS(rand, Float64, length(active), psp, virtual; left, right = oneunit(left))
end

dmrg(ψ, ham, D; sweeps = 20) = first(find_groundstate(ψ, ham, DMRG2(; trscheme = truncrank(D), verbosity = 0, maxiter = sweeps, miniter = sweeps), disk_environments(ψ, ham)))
energy(ψ, ham) = real(expectation_value(ψ, ham))

ham = qchem_mpo(ints);

## 1. Reduced density matrices

`qchem_rdms(ψ)` returns $(dV, dK)$ in the convention of `fused_quantum_chemistry_hamiltonian`, so the energy is reproduced exactly from the integrals.
At an exact eigenstate the orbital gradient has to vanish (generalized Brillouin theorem); D = 128 is exact for this system.

We need the rdms to get the gradients with respect to the orbital rotations. To check if they're correct, we can again use these to recompute the energy, and it should match.

Convergence is a bit eratic, but generally trends in the right direction when increasing D.

In [2]:
for D in (8, 16, 32, 64, 128)
    ψ = dmrg(random_mps(), ham, D)
    (dV, dK) = qchem_rdms(ψ)
    g = MPSKitExperimental.orbital_gradient(ints, dV, dK)
    println("D = $D:  E(mpo) = ", energy(ψ, ham), "   E(rdm) = ", rdm_energy(ints, dV, dK),
            "   N = ", round(tr(dK), digits = 10), "   |orbital gradient| = ", norm(g))
end

D = 8:  E(mpo) = -107.17070796123896   E(rdm) = -107.17070796123946   N = 14.0   |orbital gradient| = 0.012244110999032604
D = 16:  E(mpo) = -107.6305569764466   E(rdm) = -107.6305569764466   N = 14.0   |orbital gradient| = 0.030286769593699515
D = 32:  E(mpo) = -107.20799941978974   E(rdm) = -107.20799941979023   N = 14.0   |orbital gradient| = 0.020187148537231796
D = 64:  E(mpo) = -107.64824411954666   E(rdm) = -107.6482441195466   N = 14.0   |orbital gradient| = 0.000556845130644473
D = 128:  E(mpo) = -107.64825097401342   E(rdm) = -107.64825097401342   N = 14.0   |orbital gradient| = 2.0173163456798805e-14


## 2. Orbital optimization

In [3]:
D = 20
ψ₀ = dmrg(random_mps(;), ham, D) 
E_dmrg = energy(ψ₀, ham)

-107.63142081811034

**DMRGSCF**

a few DMRG sweeps in the current orbitals, then optimize the orbitals completely at fixed RDMs, and repeat.

In [4]:
t = @elapsed (ψ_ds, U_ds, E_ds) = find_groundstate(ψ₀, ints,
    DMRGSCF(; mps_alg = DMRG2(; trscheme = truncrank(D), verbosity = 0, maxiter = 2,miniter = 2), maxiter = 10, tol = 1e-10))
println("DMRGSCF: E = $E_ds   ($(round(t, digits = 1)) s)")

┌ Info: DMRGSCF 1: E(mps) = -107.63142373306408, E(orbitals) = -107.63148757931084, |∇U| = 0.005525743427302532
└ @ MPSKitExperimental /home/maarten/projects/MPSKitExperimental.jl/src/quantumchemistry/grassmann_scf.jl:1341
┌ Info: DMRGSCF 2: E(mps) = -107.63149035251105, E(orbitals) = -107.631490391606, |∇U| = 0.000186109854965152
└ @ MPSKitExperimental /home/maarten/projects/MPSKitExperimental.jl/src/quantumchemistry/grassmann_scf.jl:1341
┌ Info: DMRGSCF 3: E(mps) = -107.63149280334443, E(orbitals) = -107.6314928381027, |∇U| = 0.00017098839090069596
└ @ MPSKitExperimental /home/maarten/projects/MPSKitExperimental.jl/src/quantumchemistry/grassmann_scf.jl:1341
┌ Info: DMRGSCF 4: E(mps) = -107.63149499237949, E(orbitals) = -107.63149502326097, |∇U| = 0.00015732636605179869
└ @ MPSKitExperimental /home/maarten/projects/MPSKitExperimental.jl/src/quantumchemistry/grassmann_scf.jl:1341
┌ Info: DMRGSCF 5: E(mps) = -107.63149694154757, E(orbitals) = -107.63149696892151, |∇U| = 0.00014499093892

DMRGSCF: E = -107.63150381981742   (51.9 s)


┌ Info: DMRGSCF 10: E(mps) = -107.63150380526007, E(orbitals) = -107.63150381981742, |∇U| = 9.800622733424253e-5
└ @ MPSKitExperimental /home/maarten/projects/MPSKitExperimental.jl/src/quantumchemistry/grassmann_scf.jl:1341


**GrassmannSCF**

Conjugate gradient on the product manifold (left-canonical MPS as points on Grassmann manifolds) × (the orbitals)

In [5]:
t = @elapsed (ψ_gs, U_gs, E_gs) = find_groundstate(ψ₀, ints, GrassmannSCF(; maxiter = 100, tol = 1e-10, verbosity = 1))
println("GrassmannSCF: E = $E_gs   ($(round(t, digits = 1)) s)")

┌ Warning: CG: not converged to requested tol after 100 iterations and time  1.23 m: f = -1.076315891655e+02, ‖∇f‖ = 7.5244e-07
└ @ OptimKit /home/maarten/.julia/packages/OptimKit/K7Ujj/src/cg.jl:188


GrassmannSCF: E = -107.63158916553782   (77.4 s)


In [6]:
println(rpad("DMRG2, D = $D", 22), E_dmrg)
println(rpad("DMRGSCF", 22), E_ds, "   gain ", round(1e3 * (E_dmrg - E_ds), digits = 3), " mEh")
println(rpad("GrassmannSCF", 22), E_gs, "   gain ", round(1e3 * (E_dmrg - E_gs), digits = 3), " mEh")
println(rpad("FCI", 22), E_FCI)

DMRG2, D = 20         -107.63142081811034
DMRGSCF               -107.63150381981742   gain 0.083 mEh
GrassmannSCF          -107.63158916553782   gain 0.168 mEh
FCI                   -107.64825097


## 3. Frozen-core CASSCF

With `active = ...` the MPS only covers the active orbitals: the orbitals before `active` are doubly occupied (folded into a mean-field $E_0$ and $K$, see `active_space`), the ones after are empty. The orbital optimization still rotates all orbitals, using the active RDMs embedded in the full space (`embed_rdms`).

In [7]:
println("diag(K) = ", round.(diag(ints.K), digits = 2))
P = Matrix(1.0I, NORB, NORB)[:, sortperm(diag(ints.K))]
println("after P = ", round.(diag(rotate_integrals(ints, P).K), digits = 2))

active = 3:NORB   # CAS(10e, 8o), both 1s orbitals frozen
cas_ham = qchem_mpo(active_space(rotate_integrals(ints, P), active))
ψ_cas = dmrg(random_mps(active), cas_ham, 50)   # exact within the active space
E_casci = energy(ψ_cas, cas_ham)

diag(K) = [-27.58, -9.58, -7.79, -7.85, -7.85, -27.58, -7.69, -8.32, -8.08, -8.08]
after P = [-27.58, -27.58, -9.58, -8.32, -8.08, -8.08, -7.85, -7.85, -7.79, -7.69]


-107.64693265994282

In [8]:
(_, _, E_casscf) = find_groundstate(ψ_cas, ints,
    DMRGSCF(; mps_alg = DMRG2(; trscheme = truncrank(50), verbosity = 0, miniter = 2, maxiter = 2), maxiter = 30, tol = 1e-10, orb_tol = 1e-8);
    active, U = P)
(_, _, E_casscf_gs) = find_groundstate(ψ_cas, ints, GrassmannSCF(; maxiter = 100, tol = 1e-10, verbosity = 1); active, U = P)

println(rpad("CASCI(10e,8o)", 30), E_casci)
println(rpad("CASSCF(10e,8o), DMRGSCF", 30), E_casscf)
println(rpad("CASSCF(10e,8o), GrassmannSCF", 30), E_casscf_gs)
println(rpad("FCI", 30), E_FCI, "   (core correlation ", round(1e3 * (E_casscf - E_FCI), digits = 3), " mEh)")

┌ Info: DMRGSCF 1: E(mps) = -107.64693265772634, E(orbitals) = -107.64710901746116, |∇U| = 0.007797601005261871
└ @ MPSKitExperimental /home/maarten/projects/MPSKitExperimental.jl/src/quantumchemistry/grassmann_scf.jl:1341
┌ Info: DMRGSCF 2: E(mps) = -107.64711830719135, E(orbitals) = -107.64712713767928, |∇U| = 0.0014624868069526237
└ @ MPSKitExperimental /home/maarten/projects/MPSKitExperimental.jl/src/quantumchemistry/grassmann_scf.jl:1341
┌ Info: DMRGSCF 3: E(mps) = -107.647135875884, E(orbitals) = -107.64714473425819, |∇U| = 0.001269420339335587
└ @ MPSKitExperimental /home/maarten/projects/MPSKitExperimental.jl/src/quantumchemistry/grassmann_scf.jl:1341
┌ Info: DMRGSCF 4: E(mps) = -107.6471536780744, E(orbitals) = -107.64716262189926, |∇U| = 0.0012691774664337309
└ @ MPSKitExperimental /home/maarten/projects/MPSKitExperimental.jl/src/quantumchemistry/grassmann_scf.jl:1341
┌ Info: DMRGSCF 5: E(mps) = -107.64717161904674, E(orbitals) = -107.64718064029687, |∇U| = 0.0012910581532349

CASCI(10e,8o)                 -107.64693265994282
CASSCF(10e,8o), DMRGSCF       -107.64778662384856
CASSCF(10e,8o), GrassmannSCF  -107.64797577460132
FCI                           -107.64825097   (core correlation 0.464 mEh)


┌ Warning: CG: not converged to requested tol after 100 iterations and time  1.04 m: f = -1.076479757746e+02, ‖∇f‖ = 4.0160e-04
└ @ OptimKit /home/maarten/.julia/packages/OptimKit/K7Ujj/src/cg.jl:188
